01. Librerias, Carga y Ejecución

In [17]:
import pandas as pd
from pathlib import Path
import warnings

# Configuración de limpieza
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# 1. Localización y Carga de Datos (Ejercicio 1)
ruta_base = Path(r"C:\Users\igcamposg\Downloads\Caso de negocio P&S\Inputs")
archivo = next((p for p in ruta_base.glob("*") if p.suffix.lower() in {".csv", ".xlsx", ".xls"}), None)

if not archivo:
    print(f"Error: No se encontró el archivo en {ruta_base}")
else:
    df = pd.read_csv(archivo) if archivo.suffix == ".csv" else pd.read_excel(archivo)
    
    print(f"--- EJERCICIO 1: Exploración ({archivo.name}) ---")
    print(df.head(5))
    print("\nResumen Numérico:")
    print(df.describe().round(2))

    # 2. Tasa de Conversión (CVR) por Hora (Ejercicio 2)
    cvr_ranking = (
        df.query("ANIO == 2024")
        .groupby("HORA_COMPRA")
        .agg(total_envios=("TOTAL_ENVIOS", "sum"), visitas=("VISITAS", "sum"))
        .assign(CVR=lambda x: (x["total_envios"] / x["visitas"]).round(4))
        .sort_values("CVR", ascending=False)
    )
    print("\n--- EJERCICIO 2: Ranking CVR por Hora (Top 5) ---")
    print(cvr_ranking.head(5))

    # 3. % On-Time Mensual Zacatecas-Oaxaca (Ejercicio 3)
    on_time_mensual = (
        df.query("ANIO == 2024 and ORIGEN == 'ZACATECAS' and DESTINO == 'OAXACA'")
        .groupby("MES")
        .apply(lambda x: (x["ENTREGA_A_TIEMPO"].sum() / x["TOTAL_ENVIOS"].sum()) * 100, include_groups=False)
        .to_frame("PORCENTAJE_ON_TIME")
        .round(2)
    )
    print("\n--- EJERCICIO 3: Performance On-Time Mensual ZAC-OAX ---")
    print(on_time_mensual)

    # 4. Análisis: Hora de Compra Ideal (Ejercicio 4)
    # Criterio: Mayor CVR cumpliendo con el límite de Delay <= 4.5%
    hora_ideal = (
        df.query("ANIO == 2024")
        .groupby("HORA_COMPRA")
        .agg(
            total_envios=("TOTAL_ENVIOS", "sum"),
            visitas=("VISITAS", "sum"),
            demoras=("ENTREGA_DEMORADA", "sum")
        )
        .assign(CVR=lambda x: (x["total_envios"] / x["visitas"]).round(4))
        .assign(pct_delay=lambda x: (x["demoras"] / x["total_envios"]).round(4))
        .query("pct_delay <= 0.045")
        .sort_values("CVR", ascending=False)
        .head(1)
    )
    print("\n--- EJERCICIO 4: Hora de Compra Ideal (Analítica) ---")
    print(hora_ideal)

--- EJERCICIO 1: Exploración (CasoNegocioPromiseSpeed Jun_25.xlsx) ---
   ANIO  MES  SEMANA  HORA_COMPRA  HORA_ENTREGA     ORIGEN DESTINO EJECUCION_ENTREGA  ENTREGA_DEMORADA  ENTREGA_ANTICIPADA  ENTREGA_A_TIEMPO  CANCELADO  TOTAL_ENVIOS  VISITAS
0  2024    3       9           11          12.0  ZACATECAS  OAXACA           ON_TIME                 0                   0                10          0            10       16
1  2024    3       9           21          15.0  ZACATECAS  OAXACA             DELAY                 1                   0                 0          0             1        2
2  2024    3       9           22          14.0  ZACATECAS  OAXACA             EARLY                 0                   9                 0          0             9       17
3  2024    3       9            9          20.0  ZACATECAS  OAXACA             EARLY                 0                   4                 0          0             4        5
4  2024    3       9           20          16.0  ZACAT